# 6. Quantum Computing Algorithms

We've built up the mathematical foundations (Chapter 2), explored wave mechanics (Chapter 3), understood qubits and gates (Chapter 4), and seen how entanglement works (Chapter 5). Now it's time to put it all together and build **quantum algorithms** — programs that run on quantum computers.

Unlike classical algorithms that manipulate bits (0s and 1s), quantum algorithms exploit superposition, interference, and entanglement to solve certain problems *exponentially faster* than any known classical algorithm.

## 6.1 Quantum Circuit Model

A **quantum circuit** is a sequence of quantum gates applied to a set of qubits, followed by measurements. The circuit is the quantum analogue of a classical logic circuit.

### Circuit conventions
- Each horizontal line represents one qubit over time (left to right)
- Gates are boxes on those lines
- Measurement is represented by a meter symbol at the end
- Classical bits (from measurement) are represented by double lines

Here's the simplest quantum circuit: prepare $|0\rangle$, apply Hadamard, measure.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define our basic gates
I = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
H = (1 / np.sqrt(2)) * np.array([[1, 1], [1, -1]], dtype=complex)
S = np.array([[1, 0], [0, 1j]], dtype=complex)
T = np.array([[1, 0], [0, np.exp(1j * np.pi / 4)]], dtype=complex)

# Basis states
ket0 = np.array([[1], [0]], dtype=complex)
ket1 = np.array([[0], [1]], dtype=complex)

def measure(ket, trials=10000):
    """Simulate measurement of a single qubit."""
    p0 = abs(ket[0, 0])**2
    results = np.random.random(trials)
    zeros = np.sum(results < p0)
    ones = trials - zeros
    return zeros / trials, ones / trials

# Create |+⟩ = H|0⟩ and measure
psi_plus = H @ ket0
p0, p1 = measure(psi_plus)

print("Circuit: |0⟩ --[H]--[M]--")
print(f"State before measurement: |+⟩ = {psi_plus.flatten()}")
print(f"P(0) = {p0:.4f}, P(1) = {p1:.4f}")
print(f"Theoretical: P(0) = P(1) = 0.5000")

## 6.2 The Deutsch-Jozsa Algorithm

The Deutsch-Jozsa algorithm (1992) was one of the first quantum algorithms that demonstrated a *provable* speedup over any classical algorithm. While the problem it solves is artificial, it beautifully illustrates how quantum parallelism and interference work.

### The Problem

Suppose you're given a **black box** (called an *oracle*) that computes a function $f(x)$ on $n$ bits. You are **promised** that the function is either:
- **Constant:** $f(x)$ returns the same value (all 0s or all 1s) for every input $x$, OR
- **Balanced:** $f(x)$ returns 0 for exactly *half* of the inputs and 1 for the other half.

Your job: determine whether $f$ is constant or balanced.

### Classical approach

In the worst case, a classical computer would need to evaluate $f$ on **more than half** of all possible inputs ($2^{n-1} + 1$ evaluations) to be certain. For $n = 50$ qubits, that's over $10^{15}$ evaluations — impossible.

### Quantum approach

The Deutsch-Jozsa algorithm needs only **one** query to the oracle, regardless of $n$! Here's how:

1. Prepare $n$ qubits in $|0\rangle$ and 1 auxiliary qubit in $|1\rangle$
2. Apply Hadamard to all $n+1$ qubits
3. Apply the oracle $U_f$
4. Apply Hadamard to the $n$ qubits
5. Measure the $n$ qubits. If all are $|0\rangle$, $f$ is **constant**. Otherwise, $f$ is **balanced**.

Let's implement this for the simplest case: $n = 1$ (a single-bit function).

In [ ]:
# Deutsch-Jozsa for n=1 (the Deutsch algorithm)
# Oracle for constant function f(x) = 0
def oracle_constant_0(psi):
    """f(x) = 0 for all x. This does nothing to our state."""
    return psi.copy()

# Oracle for constant function f(x) = 1
def oracle_constant_1(psi):
    """f(x) = 1 for all x. Flips the ancilla qubit."""
    # Apply X to the ancilla (second qubit)
    X_on_ancilla = np.kron(I, X)
    return X_on_ancilla @ psi

# Oracle for balanced function f(x) = x (identity)
def oracle_balanced_id(psi):
    """f(x) = x. CNOT with control on first qubit, target on ancilla."""
    CNOT = np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]], dtype=complex)
    return CNOT @ psi

# Oracle for balanced function f(x) = NOT x
def oracle_balanced_not(psi):
    """f(x) = NOT x. X on control, then CNOT, then X on control."""
    X_on_first = np.kron(X, I)
    CNOT = np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]], dtype=complex)
    return X_on_first @ CNOT @ X_on_first @ psi

def deutsch_algorithm(oracle):
    """Run the Deutsch algorithm for a given oracle."""
    # Start with |0⟩|1⟩
    psi = np.kron(ket0, ket1)
    
    # Apply H to both qubits: H⊗H
    H2 = np.kron(H, H)
    psi = H2 @ psi
    
    # Apply oracle
    psi = oracle(psi)
    
    # Apply H to first qubit only: H⊗I
    H_on_first = np.kron(H, I)
    psi = H_on_first @ psi
    
    # Measure the first qubit
    p0 = abs(psi[0,0])**2 + abs(psi[1,0])**2
    return "Constant" if p0 > 0.99 else "Balanced"

print("Testing Deutsch algorithm:")
print(f"  f(x) = 0  → {deutsch_algorithm(oracle_constant_0)} (expected: Constant)")
print(f"  f(x) = 1  → {deutsch_algorithm(oracle_constant_1)} (expected: Constant)")
print(f"  f(x) = x  → {deutsch_algorithm(oracle_balanced_id)} (expected: Balanced)")
print(f"  f(x) = ¬x → {deutsch_algorithm(oracle_balanced_not)} (expected: Balanced)")
print("\nOne query to the oracle. Classical would need at worst 2 queries!")

### Why does this work?

The key is **interference**. The Hadamard gates before and after the oracle create a pattern of constructive and destructive interference:

- If $f$ is constant, all the probability amplitudes **constructively interfere** back into the $|0\rangle$ state.
- If $f$ is balanced, the amplitudes **destructively interfere** at $|0\rangle$, ensuring we measure $|1\rangle$ (or some other non-zero state).

> **The big idea:** The quantum computer explores *all* possible inputs simultaneously using superposition (quantum parallelism), then uses interference to cancel out the "wrong" answers and amplify the "right" one. This pattern — parallel evaluation + interference — is the heart of most quantum algorithms.

## 6.3 Grover's Search Algorithm

Grover's algorithm (1996) solves the problem of **searching an unsorted database**. Classically, finding a specific item in a list of $N$ items takes up to $N$ checks (linear search). Grover's algorithm does it in only $\sqrt{N}$ operations — a *quadratic* speedup.

> **Why does this matter?** For a database of 1 million items, classical search takes up to 1 million checks. Grover's algorithm needs only about 1000 — that's 1000x faster! While not as dramatic as the *exponential* speedup of Shor's algorithm, a quadratic speedup is still enormous for large datasets.

### The algorithm

1. Prepare $n$ qubits in $|0\rangle^{\otimes n}$ (where $N = 2^n$ states)
2. Apply Hadamard to all qubits to create equal superposition
3. Repeat $\approx \frac{\pi}{4}\sqrt{N}$ times:
   - Apply the **oracle** (marks the target state by flipping its phase)
   - Apply the **diffusion operator** (inversion about the mean)
4. Measure — the result is the target state with high probability

The oracle flips the sign of the target state. The diffusion operator then amplifies that state's amplitude while shrinking the others. Repeating this gradually concentrates probability onto the target.

In [ ]:
def grover_oracle(n, target):
    """Create an oracle that marks the target state.
    Oracle applies -1 to |target⟩ and leaves all other basis states unchanged.
    """
    N = 2**n
    oracle = np.eye(N, dtype=complex)
    oracle[target, target] = -1
    return oracle

def diffusion_operator(n):
    """The Grover diffusion operator: inversion about the mean."""
    N = 2**n
    # D = 2|ψ⟩⟨ψ| - I where |ψ⟩ = H^⊗n|0⟩
    D = (2 / N) * np.ones((N, N), dtype=complex)
    D -= np.eye(N, dtype=complex)
    return D

def grover_search(n, target):
    """Run Grover's search algorithm."""
    N = 2**n
    
    # Initialise: |0⟩^⊗n
    psi = np.zeros((N, 1), dtype=complex)
    psi[0, 0] = 1
    
    # Apply H^⊗n
    H_n = 1
    for _ in range(n):
        H_n = np.kron(H_n, H)
    if n == 2:
        H_n = np.kron(H, H)
    elif n == 3:
        H_n = np.kron(np.kron(H, H), H)
    psi = H_n @ psi
    
    # Grover iterations: ~π/4 * sqrt(N) times
    num_iterations = int(np.pi / 4 * np.sqrt(N))
    if num_iterations < 1:
        num_iterations = 1
    
    oracle = grover_oracle(n, target)
    diffuser = diffusion_operator(n)
    
    for _ in range(num_iterations):
        psi = oracle @ psi
        psi = diffuser @ psi
    
    return psi, num_iterations

# Test with 3 qubits (N=8) looking for target=5
n_qubits = 3
target = 5
result, iters = grover_search(n_qubits, target)

# Find most probable outcome
probabilities = np.abs(result.flatten())**2
most_likely = np.argmax(probabilities)
prob_target = probabilities[target]

print(f"Grover's algorithm with {n_qubits} qubits (N={2**n_qubits} items)")
print(f"Target item: {target}")
print(f"Number of Grover iterations: {iters}")
print(f"Classical search requires up to {2**n_qubits} checks")
print(f"\nMost likely outcome: |{most_likely}⟩ (P = {probabilities[most_likely]:.4f})")
print(f"Probability of measuring target |{target}⟩: {prob_target:.4f}")
print(f"\nAll outcome probabilities:")
for i, p in enumerate(probabilities):
    marker = " ← target" if i == target else ""
    print(f"  |{i}⟩: P = {p:.4f}{marker}")

### Visualising Grover's Amplitude Amplification

The magic of Grover's algorithm is easiest to see when we plot how the amplitudes evolve. Each iteration of (oracle + diffusion) rotates the state vector closer to the target.

In [ ]:
def grover_iterations_visual(n, target):
    """Track probability of target state across Grover iterations."""
    N = 2**n
    
    # Initialise
    psi = np.zeros((N, 1), dtype=complex)
    psi[0, 0] = 1
    
    # Hadamard on all
    H_n = 1
    for _ in range(n):
        H_n = np.kron(H_n, H)
    psi = H_n @ psi
    
    oracle = grover_oracle(n, target)
    diffuser = diffusion_operator(n)
    
    max_iters = int(np.pi / 4 * np.sqrt(N)) + 5
    probs = []
    
    for i in range(max_iters):
        psi = oracle @ psi
        psi = diffuser @ psi
        p_target = abs(psi[target, 0])**2
        probs.append(p_target)
    
    return probs

probs_evolution = grover_iterations_visual(3, 5)

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(probs_evolution) + 1), probs_evolution, 'bo-', linewidth=2, markersize=6)
plt.axhline(y=1.0, color='gray', linestyle='--', alpha=0.7, label='Perfect probability (1.0)')
plt.axhline(y=0.125, color='red', linestyle=':', alpha=0.7, label='Initial uniform (1/N)')
plt.xlabel('Grover Iteration', fontsize=12)
plt.ylabel('Probability of Target State', fontsize=12)
plt.title('Grover Search: Probability of Target State vs Iterations (n=3, target=|5⟩)', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Optimal iterations: {int(np.pi/4 * np.sqrt(8))}")
print(f"Peak probability: {max(probs_evolution):.4f} at iteration {probs_evolution.index(max(probs_evolution)) + 1}")

The plot shows the probability oscillating — it goes up to near 1, then back down, like a sine wave. The optimal stopping point is when the probability peaks, which occurs after roughly $\frac{\pi}{4}\sqrt{N}$ iterations. If you run too many iterations, the probability *decreases* — you've overshot!

> **Key insight:** Grover's algorithm doesn't give the answer with 100% certainty — it gives it with *high probability*. We can repeat the algorithm a few times to amplify confidence. This trade-off (fewer oracle calls vs probabilistic answer) is a common pattern in quantum algorithms.

## 6.4 Quantum Fourier Transform

The **Quantum Fourier Transform (QFT)** is the quantum analogue of the classical discrete Fourier transform. It is a crucial building block for Shor's factoring algorithm and many other quantum algorithms.

The QFT maps a quantum state $|x\rangle$ to:
$$\text{QFT}|x\rangle = \frac{1}{\sqrt{N}} \sum_{y=0}^{N-1} e^{2\pi i x y / N} |y\rangle$$

This looks complicated, but it can be built using only Hadamard gates and controlled phase gates. The key insight is that the QFT can be implemented in only $O(n^2)$ gates (where $n$ is the number of qubits), compared to $O(n 2^n)$ for the classical FFT — an *exponential* speedup!

In [ ]:
def QFT(psi):
    """Quantum Fourier Transform on 3 qubits."""
    n = int(np.log2(len(psi)))
    
    if n != 3:
        raise ValueError("This implementation works for 3 qubits")
    
    N = len(psi)
    
    # Controlled phase gates
    R_k = lambda k: np.array([[1, 0], [0, np.exp(2j * np.pi / (2**k))]])
    
    # Build the QFT matrix for 3 qubits
    QFT_matrix = np.zeros((N, N), dtype=complex)
    for x in range(N):
        for y in range(N):
            QFT_matrix[x, y] = np.exp(2j * np.pi * x * y / N) / np.sqrt(N)
    
    return QFT_matrix @ psi

def inverse_QFT(psi):
    """Inverse QFT on 3 qubits."""
    N = len(psi)
    QFT_matrix = np.zeros((N, N), dtype=complex)
    for x in range(N):
        for y in range(N):
            QFT_matrix[x, y] = np.exp(-2j * np.pi * x * y / N) / np.sqrt(N)
    
    return QFT_matrix @ psi

# Demonstrate QFT on a basis state |3⟩
n = 3
N = 2**n
psi = np.zeros((N, 1), dtype=complex)
psi[3, 0] = 1  # State |3⟩ = |011⟩

qft_result = QFT(psi)
qft_probs = np.abs(qft_result.flatten())**2

print("QFT on |3⟩ (|011⟩) for 3 qubits:")
print("Output state amplitudes:")
for i, amp in enumerate(qft_result.flatten()):
    print(f"  |{i}⟩: {amp.real:+.4f} + {amp.imag:+.4f}i  (P = {qft_probs[i]:.4f})")

# Verify QFT† · QFT = I (unitarity)
identity_check = inverse_QFT(qft_result)
print(f"\nQFT† · QFT |3⟩ = |3⟩? {np.abs(identity_check[3, 0]) > 0.99}")

## 6.5 Shor's Algorithm (Conceptual)

Shor's algorithm (1994) is perhaps the most famous quantum algorithm. It can factor large numbers exponentially faster than the best known classical algorithm. This is practically important because the security of RSA encryption — the foundation of secure Internet communication — relies on the difficulty of factoring large numbers.

### The algorithm in outline

Shor's algorithm factors an integer $N$ by finding the **period** of the modular exponential function $f(x) = a^x \bmod N$:

1. Pick a random number $a < N$
2. Find the period $r$ such that $a^r \equiv 1 \pmod{N}$ using QFT
3. If $r$ is even, compute $\gcd(a^{r/2} \pm 1, N)$ to extract factors

Step 2 is where the quantum speedup comes from. A classical computer would need to try $O(N)$ values of $x$ to find the period. Shor's algorithm uses the QFT to find the period in $O((\log N)^3)$ steps — an *exponential* speedup.

> **Why does the QFT find periods?** The Fourier transform maps a periodic function to a peak at the frequency corresponding to the period. The QFT does this in superposition, extracting the period from the quantum state in a single measurement.

Let's simulate a simplified version for factoring $N = 15$ (the smallest number that demonstrates the algorithm).

In [ ]:
def shor_simulate(N=15, a=7):
    """Simulate Shor's algorithm for finding the period.
    Uses a classical search for the period (simulating what QFT would find).
    """
    print(f"Factoring N = {N}")
    print(f"Randomly chosen a = {a}")
    
    # Check gcd(a, N) first
    import math
    gcd_an = math.gcd(a, N)
    if gcd_an > 1:
        print(f"Lucky! gcd({a}, {N}) = {gcd_an} is a nontrivial factor.")
        return gcd_an
    
    # Find the period r of f(x) = a^x mod N
    # In a real quantum computer, this uses QFT.
    # Here we brute force to demonstrate the logic.
    print("\nFinding period r of f(x) = a^x mod N...")
    x = 1
    r = None
    for r_candidate in range(1, N):
        if pow(a, r_candidate, N) == 1:
            r = r_candidate
            break
    
    if r is None or r % 2 != 0:
        print(f"Period r = {r} (odd or failed), try different a")
        return None
    
    print(f"Period found: r = {r}")
    print(f"Check: {a}^{r} mod {N} = {pow(a, r, N)} (correct: = 1)")
    
    # Compute factors
    factor1 = math.gcd(pow(a, r // 2, N) - 1, N)
    factor2 = math.gcd(pow(a, r // 2, N) + 1, N)
    
    print(f"\nComputing factors:")
    print(f"  gcd({a}^(r/2) - 1, N) = gcd({pow(a, r//2, N)} - 1, {N}) = {factor1}")
    print(f"  gcd({a}^(r/2) + 1, N) = gcd({pow(a, r//2, N)} + 1, {N}) = {factor2}")
    
    factors = set()
    if factor1 > 1 and factor1 < N:
        factors.add(factor1)
    if factor2 > 1 and factor2 < N:
        factors.add(factor2)
    
    if factors:
        print(f"\n✅ Success! Factors of {N}: {factors}")
        print(f"Verification: {sorted(factors)[0]} × {sorted(factors)[1]} = {sorted(factors)[0] * sorted(factors)[1]}")
    else:
        print("\n❌ Failed to find nontrivial factors. Try different a.")
    
    return factors

shor_simulate(15, 7)

> **Note:** The code above uses a classical brute-force search to find the period $r$. In a real quantum computer, the period would be found using QFT in superposition. The structure of the algorithm — compute $f(x)$, apply QFT, measure to extract the period — is identical; only the *method* for finding the period differs.

## 6.6 Quantum Error Correction (Conceptual)

Real quantum computers are noisy. Qubits decohere (lose their quantum state) over time, gates are imperfect, and measurements are noisy. Without error correction, large-scale quantum computation would be impossible.

### The challenge

Quantum error correction is fundamentally harder than classical error correction because:
1. **No cloning theorem:** You cannot copy a quantum state, so you can't simply triple-redundancy as in classical systems.
2. **Continuous errors:** Qubits can suffer from *any* small rotation, not just bit flips.
3. **Measurement destroys superposition:** Measuring a qubit to check for errors collapses its state.

### The solution: the 3-qubit bit-flip code

The simplest quantum error correction code protects against bit-flip errors (X gates applied by noise). It works by *encoding* one logical qubit into three physical qubits:

$$|0_L\rangle = |000\rangle, \quad |1_L\rangle = |111\rangle$$

This is analogous to the classical repetition code. If a single qubit flips, we can detect it by measuring the parity (without measuring individual qubits' values) and correct it.

In [ ]:
def encode_bitflip(psi_original):
    """Encode one logical qubit into three physical qubits.
    Uses two CNOT gates: |ψ⟩|00⟩ → |ψ⟩|ψψ⟩
    """
    # Start with |ψ⟩|00⟩
    psi = np.kron(psi_original, np.kron(ket0, ket0))
    
    # CNOT from qubit 0 to qubit 1 (control on 0, target on 1)
    # This is CNOT ⊗ I on the 3-qubit space
    CNOT_01 = np.kron(np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]], dtype=complex), I)
    psi = CNOT_01 @ psi
    
    # CNOT from qubit 0 to qubit 2 (control on 0, target on 2)
    CNOT_02 = np.kron(I, np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]], dtype=complex))
    CNOT_02 = CNOT_02 @ np.array([
        [1,0,0,0,0,0,0,0],
        [0,1,0,0,0,0,0,0],
        [0,0,1,0,0,0,0,0],
        [0,0,0,1,0,0,0,0],
        [0,0,0,0,0,1,0,0],
        [0,0,0,0,1,0,0,0],
        [0,0,0,0,0,0,0,1],
        [0,0,0,0,0,0,1,0]
    ], dtype=complex)
    # Simpler approach: just build the 8x8 encoding matrix
    encode_matrix = np.zeros((8, 2), dtype=complex)
    encode_matrix[0, 0] = 1  # |0⟩ → |000⟩
    encode_matrix[7, 1] = 1  # |1⟩ → |111⟩
    
    encoded = encode_matrix @ psi_original
    return encoded

def apply_bitflip_error(psi, qubit):
    """Apply an X error to a specific qubit."""
    if qubit == 0:
        error = np.kron(np.kron(X, I), I)
    elif qubit == 1:
        error = np.kron(np.kron(I, X), I)
    elif qubit == 2:
        error = np.kron(np.kron(I, I), X)
    else:
        raise ValueError(f"Invalid qubit: {qubit}")
    return error @ psi

def measure_syndrome_and_correct(psi):
    """Detect and correct a single bit-flip error.
    Syndrome: measure Z⊗Z and Z⊗Z on pairs of qubits.
    Since we're simulating, we directly check the state.
    """
    # Get the probability distribution
    probs = np.abs(psi.flatten())**2
    
    # The encoded states |000⟩ (idx 0) and |111⟩ (idx 7) are valid
    # If we detect something else, apply correction
    error_qubit = None
    for idx, p in enumerate(probs):
        if p > 0.99:
            bits = f"{idx:03b}"
            # Check if all bits are same (correct)
            if bits == "000" or bits == "111":
                return psi, None  # No error
            else:
                # Find which qubit is flipped by comparing to majority
                ones = bits.count('1')
                if ones == 1:
                    error_qubit = bits.index('1')
                elif ones == 2:
                    error_qubit = bits.index('0')
                break
    
    if error_qubit is not None:
        # Apply X to correct
        corrected = apply_bitflip_error(psi, error_qubit)
        return corrected, error_qubit
    return psi, None

# Test encoding |1⟩, introducing an error, and correcting
logical_psi = ket1  # Encode |1⟩
encoded = encode_bitflip(logical_psi)

# Apply error on qubit 1
corrupted = apply_bitflip_error(encoded, 1)

# Correct
corrected, error_loc = measure_syndrome_and_correct(corrupted)

print("3-Qubit Bit Flip Error Correction:")
print(f"Original logical state: |1⟩ encoded as |111⟩")
print(f"Encoded state: |000⟩? {abs(encoded[0,0]) > 0.99} | |111⟩? {abs(encoded[7,0]) > 0.99}")
print(f"\nError applied on qubit 1 (X gate)")
print(f"Corrupted state: |{np.argmax(np.abs(corrupted.flatten())):03b}⟩")
print(f"")
print(f"Error detected on qubit: {error_loc}")
corr_idx = np.argmax(np.abs(corrected.flatten()))
print(f"Corrected state: |{corr_idx:03b}⟩")
print(f"Correction successful: {corr_idx == 7}")

> **Why this matters:** The 3-qubit bit-flip code is the foundation upon which more sophisticated codes (like the surface code used by Google and IBM) are built. These codes use many physical qubits to create one highly protected logical qubit. Current estimates suggest we need about 1000 physical qubits per logical qubit for practical quantum computing.

## 6.7 Limitations of Real Quantum Computers

| Challenge | Description | Mitigation |
|-----------|-------------|------------|
| **Decoherence** | Qubits lose their quantum state over time (typically microseconds) | Error correction, faster gates |
| **Gate infidelity** | Quantum gates are imperfect (99.9% is state-of-the-art) | Error correction, better hardware |
| **Limited connectivity** | Not all qubits can directly interact | SWAP gates, compiler optimizations |
| **Measurement errors** | Reading a qubit can disturb its state | Repeated measurements, error mitigation |
| **NISQ era** | Noisy, Intermediate-Scale Quantum (50-1000 qubits) | Hybrid quantum-classical algorithms |

## 6.8 Interactive Exercises

### Exercise 1: Grover on 2 qubits
Run Grover's algorithm for $n=2$ qubits (4 items). How many iterations are optimal? What's the success probability? Compare with the 3-qubit case.

### Exercise 2: Design your own oracle
Create a Grover oracle that marks *two* target states simultaneously (e.g., |3⟩ and |5⟩). How does Grover's algorithm perform when searching for multiple items?

### Exercise 3: QFT on different basis states
Apply the QFT to $|0\rangle$, $|1\rangle$, $|2\rangle$, and $|7\rangle$ on 3 qubits. What pattern do you observe in the output amplitudes?

### Exercise 4: The Deutsch-Jozsa oracle
Create a balanced oracle for $n=2$ that implements $f(x_1, x_2) = x_1 \oplus x_2$ (XOR). Does Deutsch-Jozsa correctly identify it as balanced?

### Exercise 5: Error correction simulation
Modify the bit-flip code simulation: apply errors to *two* qubits simultaneously. Can the 3-qubit code still correct the state? Why or why not?

### Challenge: All 4 Bell states via circuits
Write a function that creates *all four* Bell states given two inputs in the computational basis. Use only H and CNOT gates. Verify each state is correctly created and orthonormal.

---

> **The big picture:** You've now seen the complete pipeline from basic math (ch02) → wave mechanics (ch03) → qubits and gates (ch04) → entanglement (ch05) → quantum algorithms (this chapter). These are the foundations of quantum computing. The algorithms here — Deutsch-Jozsa, Grover's search, QFT, Shor's factoring, and error correction — represent the core ideas that researchers are building on to create the quantum computers of tomorrow.